# Brainstorming and Focus Group Quantitative Experimentation 1: **General US population** under **action correction** + **divergence intervention**

Can we use TinyTroupe to brainstorm product ideas?

In [1]:
import sys

from pprint import pprint

import tinytroupe
from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.experimentation import InPlaceExperimentRunner
from tinytroupe.steering import Intervention
from tinytroupe.examples import *
from tinytroupe.validation import propositions
from tinytroupe.extraction import ResultsExtractor
from tinytroupe.utils.parallel import parallel_map_dict, parallel_map_cross
from tinytroupe.validation import hard_persona_adherence, persona_adherence, self_consistency, fluency, task_completion, divergence

# specific utilities
from common_utils import *


!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inaccurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!

Looking for default config on: C:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\tinytroupe\utils\..\config.ini
Found custom config on: c:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\publications\paper_artifacts_april-2026\config.ini
TinyTroupe version: 0.8.0
Current date and time (local): 2026-04-27 22:09:08
Current date and time (UTC):   2026-04-28 01:09:08

Current TinyTroupe configuration 
[OpenAI]
api_type = azure
azure_api_version = 2024-12-01-preview
model = gpt-5-mini
reasoning_model = o3-mini
vision_detail = auto
embedding_model = text-embedding-3-small
azure_embedding_model_api_version = 2023-05-15
max_completion_tokens = 128000
timeout = 300
max_attempts = 5
waiting_tim

## Parameters

In [2]:
full_mode = True  # set to True to run the full mode with all agents and tasks

# avoid displaying the communication, to make the output cleaner for eval
TinyPerson.communication_display = False

In [3]:
if full_mode:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 12
    qty_proposals = 4

else:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 4
    qty_proposals = 1


## Experiment setup

In [4]:
experiment_runner = InPlaceExperimentRunner("./brainstorming_and_focus_group_quantitative_experimentation_1.json")

experiment_runner.add_experiment("Control")
experiment_runner.add_experiment("Treatment")

In [5]:
experiment_runner.activate_next_experiment()

#experiment_runner.fix_active_experiment("Control")
#experiment_runner.fix_active_experiment("Treatment")

In [6]:
print(f"Running experiment {experiment_runner.get_active_experiment()}")

Running experiment Control


## Agents and populations

In [7]:

people = []
if not experiment_runner.has_finished_all_experiments():
    # load agents
    people = TinyPerson.load_specifications_from_folder("./population/usa_general_2")

    # filter to make it go faster?
    if qty_agents is not None:
        people = people[:qty_agents]

    # customize and print minibios 
    for person in people:
        ### person.import_fragment("./fragments/picky_customer.agent.fragment.json")
        print(person.minibio(extended=False))


Ariana Martinez-Brown is a 34 year old Kitchen Lead / Line Cook, American, currently living in Atlanta, Georgia (urban neighborhood, near a mixed commercial-residential corridor).
Ava Sinclair is a 21 year old Junior Software Developer (Front-end) / Freelance Game Designer and Graphic Designer, American, currently living in Chicago, Illinois (urban, near Bronzeville neighborhood).
Benjamin Hartley is a 42 year old Senior Engineering Manager (Director-level responsibilities), American, currently living in San Francisco, California, USA.
Carmen Alvarez-Johnson is a 59 year old Community Health Outreach Coordinator / Part-time Licensed Practical Nurse (LPN), American, currently living in Atlanta, Georgia, USA (urban neighborhood, Eastside near a community health clinic and public transit).
Carolyn Whitman is a 61 year old Senior Project Manager — Civil Infrastructure, American, currently living in Madison, Wisconsin (mid-sized Midwestern city).
Charlotte Mercer is a 14 year old Student; J

In [8]:
len(people)

12

In [9]:
# divide people in several groups of 5
people_groups = []
for i in range(0, len(people), 5):
    people_groups.append(people[i:i+5]
    )

len(people_groups)

3

In [10]:
# The experiment refers to customers

if experiment_runner.get_active_experiment() == "Control":
    for person in people:
        person.action_generator.enable_reasoning_step = False
        person.action_generator.enable_quality_checks = False

elif experiment_runner.get_active_experiment() == "Treatment":    
    for person in people:
       person.action_generator.enable_reasoning_step = False
       person.action_generator.enable_quality_checks = True
       person.action_generator.max_attempts = 5
       person.action_generator.enable_regeneration = True
       person.action_generator.quality_threshold = 5

## Proposals

In [11]:
proposals = [
    {"theme": "Food and Nutrition (food itself, consumption, preparation, transportation, storage)", 
    "objective": "ideas for new food products, either new foods, food services, food experiences, "+\
                "or food preparation tools." },
    {"theme": "Travel and Tourism (travel, tourism, hospitality, leisure)",
    "objective": "ideas for new travel and tourism services, experiences, or products." },
    {"theme": "Health and Wellbeing (health, wellness, fitness, beauty)",
    "objective": "ideas for new health and wellbeing services, experiences, or products." },
    {"theme": "Economics and Finance (economics, finance, business, work)",
    "objective": "ideas for new economic and financial services, experiences, or products." },
    {"theme": "Technology and Innovation (technology, innovation, science, research)",
    "objective": "ideas for new technology and innovation services, experiences, or products." }
]

if not full_mode:
    proposals = proposals[:qty_proposals]

## Auxiliary functions

In [12]:
def brainstorming_battery(agents, proposals, interventions, agent_propositions, environment_propositions, 
                          repetitions = 5, simulation_steps=10): 
    
    agent_propositions_scores = {}
    environment_propositions_scores = {}

    experiments_count = 0
    total_expected_experiments = len(proposals) * repetitions #* len(agents)

    # TODO remove?
    #
    # Add intervention to prevent agents from being too quiet.
    #for agent in agents:
    #    intervention = \
    #        Intervention(agent)\
    #            .set_propositional_precondition(propositions.quiet_recently)\
    #            .set_effect(lambda target: target.think("""
    #                                                    I will say something now, I've been too quiet for a while. If I am uncomfortable, 
    #                                                    or can't think of a proper response,
    #                                                    I can always say something like "I don't want to talk about this",
    #                                                    or propose another topic.
    #                                                    """))
    #    interventions.append(intervention)

    # loop over proposals and repetitions
    for proposal in proposals:

        objective = proposal["objective"]
        theme = proposal["theme"]

        for i in range(repetitions):
            print("\n############## STARTING A NEW RESEARCH SESSION #################")
            print(f"Overall experiment number: {experiments_count+1} / {total_expected_experiments}")
            print(f"Discussion objective: {objective}")
            print(f"Trial number: {i+1}")
            print(f"Agents: {agents}")

            # clear the episodic memory of all agents
            for person in agents:
                person.clear_episodic_memory()

            world = TinyWorld(agents=agents, interventions=interventions)
            
            # Participants introduce themselves
            world.broadcast(f"""
                Hello everyone! Let's start by introducing ourselves, and mentioning problems we face in our daily personal
                and professional lives related to the following theme: {theme}
                
                Please:
                  - present yourself and your background;
                  - present some key personal problems related to the theme;
                  - present some key problems related to the theme that you face in your work;
                  - present some key problems related to the theme that you see in your industry as a whole.
                  
                Don't discuss solutions yet, just the problems you face and see others facing.
                """)
            world.run(1)
            
            # now to the brainstorming session itself
            world.broadcast(f"""
                Folks, your mission is to brainstorm {objective}. 
                Please follow these guidelines:
                  - give a unique and informative name to each idea you propose, so that it is easy to refer to it. Say it like "Idea name: '<name of the idea>'".;
                  - explain why you think it is a good idea, and what problem it solves, and how you feel about it;
                  - your ideas should be new complete, self-contained, products or services, not features for other existing products or services;
                  - think of creative ideas that would somehow help you in both in your personal and professional lives.
                  - create as many different and unique ideas as you can during the brainstorming session. Each idea must be **completely** different from the others 
                    (either by yourself or by others), and not just a variation of an existing idea.
                    and not just a variation of an existing idea.
                  - you should criticize each other's ideas, in order to make sure they are as
                    good as possible, but no more than once per idea.
                  - you should also provide suggestions for improvement to each other's ideas, in order to make them as good as possible, 
                    but no more than once per idea.
                  - regardless of critique or complement, you **must** primarily propose new ideas quickly, 
                    not just build on existing ones. 
                  - propose one idea at a time, instead of proposing multiple ideas at once, to allow appropriate discussion.
                  - you should **not** propose ideas that are too similar to each other, or to the ones already proposed by others.
                  - before saying anything, THINK deeply about yourself, your beliefs, interests, needs, life, etc., to come up with ideas that are
                    truly unique and different from the ones already proposed by others.
                   
                Please start the discussion now.
                """)
            world.run(simulation_steps)

            # extract and count ideas
            rapporteur = agents[0]  # the first agent is the rapporteur
            rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
            ideas = ResultsExtractor().extract_results_from_agent(rapporteur, 
                                    extraction_objective="Consolidates the ideas that the group came up with, explaining each idea as an item of a list." \
                                                        "Add information about: what problem the idea solves; to which target audience it is meant." \
                                                        "how is it different from competing, existing, products.", 
                                    situation="A focus group to brainstorm new product ideas.",
                                    fields= ["name", "description", "problem", "target_audience", "competition_analysis"],
                                    fields_hints={"ideas": "must be the root of the resulting dictionary."},)
            pprint(ideas)
            if "ideas_qty" not in environment_propositions_scores:
                environment_propositions_scores["ideas_qty"] = []
            if ideas is not None and "ideas" in ideas and isinstance(ideas["ideas"], list):
                environment_propositions_scores["ideas_qty"].append(len(ideas["ideas"]))

            # Evaluate environment propositions in parallel
            env_results = parallel_map_dict(
                environment_propositions,
                lambda item: item[1].copy().score(
                    world, 
                    claim_variables={"task_description": f"A brainstorming or focus group session was run about: {objective}."}, 
                    return_full_response=True
                )
            )
            
            # Process environment results
            for k, result in env_results.items():
                if k not in environment_propositions_scores:
                    environment_propositions_scores[k] = []
                environment_propositions_scores[k].append(result["value"])
                print("value: ", result["value"])
                print("justification: ", result["justification"])
                print("reasoning: ", result["reasoning"])

            # Evaluate agent propositions across all agents in parallel
            agent_results = parallel_map_cross(
                [agents, agent_propositions.items()],
                lambda agent, prop_item: (
                    prop_item[0],  # proposition key
                    prop_item[1].copy().score(agent, return_full_response=True)  # result
                )
            )
            
            # Process agent results
            for k, result in agent_results:
                if k not in agent_propositions_scores:
                    agent_propositions_scores[k] = []
                if result is not None:
                    agent_propositions_scores[k].append(result["value"])
                    print("value: ", result["value"])
                    print("justification: ", result["justification"])
                    print("reasoning: ", result["reasoning"])
                    print("\n\n")
                else:
                    print(f"*****WARNING:***** Agent did not respond to proposition {k}.")
            #
            ##for k, proposition in agent_propositions.items():
            ##    for person in world.agents:
            ##        result = proposition.copy().score(person, return_full_response=True)
            ##        
            ##        if k not in agent_propositions_scores:
            ##            agent_propositions_scores[k] = []
            ##        agent_propositions_scores[k].append(result["value"])
            ##
            ##        print("value: ", result["value"])
            ##        print("justification: ", result["justification"])
            ##        print("reasoning: ", result["reasoning"])
            ##        print("\n\n")
            ##
            
            experiments_count += 1
            print("\n\n")

    return agent_propositions_scores, environment_propositions_scores



## Perform experiment

In [13]:
agent_propositions_scores={}
environment_propositions_scores={}

In [14]:
def brainstorm(people):
    global agent_propositions_scores, environment_propositions_scores
    if not experiment_runner.has_finished_all_experiments():

        interventions = []
        if experiment_runner.get_active_experiment() == "Treatment":
            interventions = \
                Intervention.create_for_each(people)\
                    .set_functional_precondition(lambda target: target.actions_count >=7)\
                    .set_textual_precondition(
                        """
                        AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS ANYMORE:
                        The last **entirely** new product/service idea proposed by this agent, if any, was proposed by him/her **more** than 10 of simulation events ago.
                        That is to say, the agent has not proposed any new product/service idea in the last 10 of his/her simulation trajectory events.
                        Additional features, variations of or other refinements to product/service ideas already proposed are NOT considered new!

                        How to compute the steps gap:
                        1. Determine the current next event number (N); and the last event number in which the agent proposed a new product/service idea (M).
                            This information can be found in the simulation trajectory.
                        2. Compute the **difference** beteween the current next event number and the last event number in which the agent proposed a new product/service idea: D = N - M
                        3. The proposition is true if, and only if, the difference D is **greater than** 10.
                        """)\
                    .set_effect(lambda target: target.think("""
                                                            I need to propose additional, **completelly** new and different, product/service ideas. This was part of the requirement for this session.
                                                            I will propose an entirely **new** idea now, I **cannot** repeat or refine previous ideas! I cannot make variations
                                                            of previous ideas (e.g., "XYZ for A", "XYZ for B", "XYZ for Z" are repetitive, there should be only one "XYZ"), 
                                                            I need to think of something **entirely** new and different.
                                                            To help me avoid repeating previous ideas, I'll now explicitly THINK about all the ideas already given by myself or
                                                            others, and then, based on that, I'll think again about a new unique idea.
                                                            """))

                                                            
        tmp_agent_propositions_scores, tmp_environment_propositions_scores = \
            brainstorming_battery(
                agents=people,
                proposals=proposals,
                interventions=interventions,    
                agent_propositions={
                    "Hard Persona Adherence": hard_persona_adherence,
                    "Self-consistency": self_consistency,
                    "Fluency": fluency
                },
                environment_propositions={
                    "Task Completion": task_completion,
                    "Divergence": divergence
                },
                repetitions=repetitions_per_task,
                simulation_steps=simulation_steps
            )

        pprint("NEW AGENT PROPOSITIONS SCORES")
        pprint(tmp_agent_propositions_scores)
        print("\n\n")
        pprint("NEW ENVIRONMENT PROPOSITIONS SCORES")
        pprint(tmp_environment_propositions_scores)

        # merge the scores lists
        agent_propositions_scores = merge_dicts_of_lists(tmp_agent_propositions_scores, agent_propositions_scores)
        environment_propositions_scores = merge_dicts_of_lists(tmp_environment_propositions_scores, environment_propositions_scores)

        return agent_propositions_scores, environment_propositions_scores

In [15]:
brainstorm(people_groups[0]) if len(people_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 10
Discussion objective: ideas for new food products, either new foods, food services, food experiences, or food preparation tools.
Trial number: 1
Agents: [TinyPerson(name='Ariana Martinez-Brown'), TinyPerson(name='Ava Sinclair'), TinyPerson(name='Benjamin Hartley'), TinyPerson(name='Carmen Alvarez-Johnson'), TinyPerson(name='Carolyn Whitman')]
2026-04-27 22:09:34,062 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 1] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 1 step 1 of 1 ─────────────────────────────────────────────

2026-04-27 22:09:34,093 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-04-27 22:09:38,722 - ThreadPoolExecutor-0_0(51560) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:09:38,737 - ThreadPoolExecutor-0_4(11948) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:09:38,742 - ThreadPoolExecutor-0_3(27736) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:09:38,863 - ThreadPoolExecutor-0_1(6780) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:09:38,879 - ThreadPoolExecutor-0_2(29532) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:09:39,288 - ThreadPoolExecutor-0_1(6780) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-27 22:09:39,292 - ThreadPoolExecutor-0_3(27736) - tinytroupe - INFO - Waiting 5.0 seconds before next API

───────────────────────────────────────────── TinyWorld 1 step 1 of 5 ─────────────────────────────────────────────

2026-04-27 22:10:51,069 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-04-27 22:10:53,534 - ThreadPoolExecutor-1_2(47752) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:10:53,564 - ThreadPoolExecutor-1_1(48000) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:10:53,602 - ThreadPoolExecutor-1_2(47752) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-27 22:10:53,625 - ThreadPoolExecutor-1_1(48000) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-27 22:10:53,644 - ThreadPoolExecutor-1_0(41116) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:10:53,651 - ThreadPoolExecutor-1_4(50652) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-27 22:10:53,651 - ThreadPoolExecutor-1_3(11456) - tinytroupe - INFO - Using A

...

──────────────────────────────────────────── TinyWorld 10 step 4 of 5 ─────────────────────────────────────────────

2026-04-28 00:30:11,874 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-28 00:30:14,323 - ThreadPoolExecutor-76_3(26236) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:30:14,347 - ThreadPoolExecutor-76_0(48664) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:30:14,375 - ThreadPoolExecutor-76_2(20112) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:30:14,398 - ThreadPoolExecutor-76_1(32944) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:30:14,408 - ThreadPoolExecutor-76_4(40048) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:30:14,455 - ThreadPoolExecutor-76_3(26236) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 00:30:14,460 - ThreadPoolExecutor-76_0(48664) - tinytroupe - INFO - Waiting 5.0 seconds befor

──────────────────────────────────────────── TinyWorld 10 step 5 of 5 ─────────────────────────────────────────────

2026-04-28 00:31:26,803 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 10] No timedelta provided, so the datetime was not advanced.
2026-04-28 00:31:28,879 - ThreadPoolExecutor-77_2(49128) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:31:28,885 - ThreadPoolExecutor-77_1(48900) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:31:28,963 - ThreadPoolExecutor-77_1(48900) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 00:31:28,971 - ThreadPoolExecutor-77_2(49128) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 00:31:29,010 - ThreadPoolExecutor-77_0(38492) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:31:29,026 - ThreadPoolExecutor-77_3(22800) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:31:29,051 - ThreadPoolExecutor-77_4(24992) - tinytroupe - INFO -

({'Hard Persona Adherence': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   5,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8],
  'Fluency': [8,
   8,
   7,
   9,
   8,
   7,
   7,
   8,
   9,
   9,
   7,
   8,
   8,
   8,
   8,
   7,
   9,
   9,
   8,
   9,
   8,
   7,
   7,
   9,
   9,
   8,
   8,
   8,
   8,
   8,
   8,
   9,
   8,
   8,
   8,
   7,
   9,
   9,
   8,
   9,
   8,
   8,
   8,
   8,
   8,
   7,
   8,
   8,
   8,
   7]},
 {'ideas_qty': [5, 5, 5, 5, 2, 5, 5, 5,

In [16]:
brainstorm(people_groups[1]) if len(people_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 10
Discussion objective: ideas for new food products, either new foods, food services, food experiences, or food preparation tools.
Trial number: 1
Agents: [TinyPerson(name='Charlotte Mercer'), TinyPerson(name='Elliot James Prescott'), TinyPerson(name='Eloise Gardner'), TinyPerson(name='Ethan Nakamura'), TinyPerson(name='Evelyn Bradford')]
2026-04-28 00:43:42,436 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 11] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 11 step 1 of 1 ─────────────────────────────────────────────

2026-04-28 00:43:42,444 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-04-28 00:43:45,288 - ThreadPoolExecutor-80_3(19032) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:43:45,307 - ThreadPoolExecutor-80_2(10160) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:43:45,362 - ThreadPoolExecutor-80_0(22532) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:43:45,374 - ThreadPoolExecutor-80_1(35264) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:43:45,398 - ThreadPoolExecutor-80_4(46500) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:43:45,403 - ThreadPoolExecutor-80_3(19032) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 00:43:45,449 - ThreadPoolExecutor-80_2(10160) - tinytroupe - INFO - Waiting 5.0 seconds befor

──────────────────────────────────────────── TinyWorld 11 step 1 of 5 ─────────────────────────────────────────────

2026-04-28 00:45:26,953 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-04-28 00:45:29,459 - ThreadPoolExecutor-81_1(3672) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:45:29,569 - ThreadPoolExecutor-81_2(26676) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:45:29,600 - ThreadPoolExecutor-81_4(34488) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:45:29,602 - ThreadPoolExecutor-81_3(5548) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:45:29,645 - ThreadPoolExecutor-81_1(3672) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 00:45:29,674 - ThreadPoolExecutor-81_0(53124) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 00:45:29,708 - ThreadPoolExecutor-81_2(26676) - tinytroupe - INFO - Waiting 5.0 seconds before n

...

──────────────────────────────────────────── TinyWorld 20 step 4 of 5 ─────────────────────────────────────────────

2026-04-28 04:05:06,652 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-04-28 04:05:10,502 - ThreadPoolExecutor-156_2(52824) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:05:10,513 - ThreadPoolExecutor-156_3(14208) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:05:10,601 - ThreadPoolExecutor-156_2(52824) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:05:10,615 - ThreadPoolExecutor-156_3(14208) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:05:10,758 - ThreadPoolExecutor-156_1(27388) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:05:10,801 - ThreadPoolExecutor-156_4(13932) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:05:10,829 - ThreadPoolExecutor-156_0(17260) - tinytroupe -

──────────────────────────────────────────── TinyWorld 20 step 5 of 5 ─────────────────────────────────────────────

2026-04-28 04:06:58,371 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 20] No timedelta provided, so the datetime was not advanced.
2026-04-28 04:07:00,838 - ThreadPoolExecutor-157_2(3740) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:07:00,856 - ThreadPoolExecutor-157_1(34692) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:07:00,862 - ThreadPoolExecutor-157_0(32904) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:07:00,932 - ThreadPoolExecutor-157_2(3740) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:07:00,948 - ThreadPoolExecutor-157_1(34692) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:07:00,958 - ThreadPoolExecutor-157_0(32904) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:07:00,958 - ThreadPoolExecutor-157

({'Hard Persona Adherence': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   5,
   9,
   9,
   7,
   9,
   7,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   7,
   9,
   9],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   5,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [17]:
brainstorm(people_groups[2]) if len(people_groups) > 2 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 10
Discussion objective: ideas for new food products, either new foods, food services, food experiences, or food preparation tools.
Trial number: 1
Agents: [TinyPerson(name='Gloria Rosario'), TinyPerson(name='Harper Sullivan')]
2026-04-28 04:21:33,049 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 21] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 21 step 1 of 1 ─────────────────────────────────────────────

2026-04-28 04:21:33,057 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-28 04:21:34,159 - ThreadPoolExecutor-160_1(23184) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:21:34,180 - ThreadPoolExecutor-160_0(17964) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:21:34,193 - ThreadPoolExecutor-160_1(23184) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:21:34,210 - ThreadPoolExecutor-160_0(17964) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:22:38,785 - ThreadPoolExecutor-160_1(23184) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-28 04:22:40,318 - ThreadPoolExecutor-160_0(17964) - httpx - INFO - HTTP 

──────────────────────────────────────────── TinyWorld 21 step 1 of 5 ─────────────────────────────────────────────

2026-04-28 04:22:40,363 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-04-28 04:22:41,532 - ThreadPoolExecutor-161_0(36936) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:22:41,563 - ThreadPoolExecutor-161_1(40808) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 04:22:41,579 - ThreadPoolExecutor-161_0(36936) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:22:41,612 - ThreadPoolExecutor-161_1(40808) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 04:23:47,877 - ThreadPoolExecutor-161_0(36936) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-28 04:23:48,249 - ThreadPoolExecutor-161_1(40808) - httpx - INFO - HTTP 

...

──────────────────────────────────────────── TinyWorld 30 step 4 of 5 ─────────────────────────────────────────────

2026-04-28 06:37:57,922 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-04-28 06:37:59,533 - ThreadPoolExecutor-236_0(15792) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 06:37:59,540 - ThreadPoolExecutor-236_1(2436) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 06:37:59,600 - ThreadPoolExecutor-236_1(2436) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 06:37:59,611 - ThreadPoolExecutor-236_0(15792) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 06:38:58,788 - ThreadPoolExecutor-236_0(15792) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-28 06:39:23,436 - ThreadPoolExecutor-236_1(2436) - httpx - INFO - HTTP Req

──────────────────────────────────────────── TinyWorld 30 step 5 of 5 ─────────────────────────────────────────────

2026-04-28 06:39:23,481 - MainThread(43852) - tinytroupe - INFO - [TinyWorld 30] No timedelta provided, so the datetime was not advanced.
2026-04-28 06:39:24,629 - ThreadPoolExecutor-237_0(28516) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 06:39:24,674 - ThreadPoolExecutor-237_0(28516) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 06:39:24,689 - ThreadPoolExecutor-237_1(38200) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-04-28 06:39:24,741 - ThreadPoolExecutor-237_1(38200) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-04-28 06:40:28,084 - ThreadPoolExecutor-237_0(28516) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-04-28 06:40:28,121 - ThreadPoolExecutor-237_0(28516) - tinytroupe - WARNING

({'Hard Persona Adherence': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   5,
   9,
   9,
   7,
   9,
   7,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   5,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [18]:
brainstorm(people_groups[3]) if len(people_groups) > 3 else None

In [19]:
brainstorm(people_groups[4]) if len(people_groups) > 4 else None

## Extract results and analyze

In [20]:
if experiment_runner.get_active_experiment() in ["Control", "Treatment"]:
    combined_scores = {**agent_propositions_scores, **environment_propositions_scores}
    experiment_runner.add_experiment_results(combined_scores, experiment_name=experiment_runner.get_active_experiment()) 
    
    plot_scores(combined_scores)

else:
    print("Experiment finished. No more experiments to run.")

{'Divergence': [0,
                8,
                4,
                2,
                4,
                2,
                2,
                2,
                5,
                9,
                6,
                7,
                3,
                1,
                3,
                8,
                3,
                2,
                3,
                1,
                0,
                2,
                0,
                0,
                0,
                0,
                0,
                0,
                2,
                0],
 'Fluency': [8,
             8,
             7,
             9,
             8,
             7,
             7,
             8,
             9,
             9,
             7,
             8,
             8,
             8,
             8,
             7,
             9,
             9,
             8,
             9,
             8,
             7,
             7,
             9,
             9,
             8,
             

,Proposition,Average Score,Standard Deviation,Count
0,Hard Persona Adherence,8.766667,0.694920,120.0
1,Self-consistency,8.900000,0.491952,120.0
2,Fluency,8.158333,0.830064,120.0
3,ideas_qty,4.310345,1.872802,29.0
4,Task Completion,9.000000,0.000000,30.0
5,Divergence,2.633333,2.684352,30.0


In [21]:
if experiment_runner.has_finished_all_experiments():
    print("All experiments have been finished.")
    print(f"STATISTICTS: Control vs")
    pprint(experiment_runner.run_statistical_tests(control_experiment_name='Control'))

    # plot scores of both experiments
    experiment_control_scores = experiment_runner.get_experiment_results("Control")
    experiment_treatment_scores = experiment_runner.get_experiment_results("Treatment")
    
    
    plot_scores(experiment_control_scores)
    plot_scores(experiment_treatment_scores)

else:
    print("Not all experiments have been finished. RESTART AND RERUN.")

Not all experiments have been finished. RESTART AND RERUN.


In [22]:
experiment_runner.finish_active_experiment()

2026-04-28 06:48:57,168 - MainThread(43852) - tinytroupe - INFO - Experiment 'Control' marked as finished.


True